### Observe Behavoir of CDF- Batch

#### Without CDF Enabled- whole file scan

In [0]:
T = "kafka_poc.engagement.cdf_off_poc"
base = "s3://retail-data-lake-dev/delta_poc/labs/cdf_off_poc"
spark.sql(f"DROP TABLE IF EXISTS {T}")
dbutils.fs.rm(base, recurse=True)

spark.sql(f"""
CREATE TABLE {T} (id INT, val STRING) USING DELTA LOCATION '{base}'
TBLPROPERTIES ('delta.enableChangeDataFeed'='false','delta.enableDeletionVectors'='false')
""")

# v1: 10 records
v1 = spark.createDataFrame(
    [(1,"A"),(2,"B"),(3,"C"),(4,"D"),(5,"E"),
     (6,"F"),(7,"G"),(8,"H"),(9,"I"),(10,"J")],
    "id INT, val STRING").repartition(1)
v1.writeTo(T).append()   # v1

# v2: 5 updates (id 1-5) + 5 inserts (id 11-15) via MERGE on id
src = spark.createDataFrame(
    [(1,"A2"),(2,"B2"),(3,"C2"),(4,"D2"),(5,"E2"),          # match -> update
     (11,"K"),(12,"L"),(13,"M"),(14,"N"),(15,"O")],         # no match -> insert
    "id INT, val STRING").createOrReplaceTempView("src")

spark.sql(f"""
MERGE INTO {T} AS t
USING src AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")   # v2



In [0]:
display(spark.sql(f"DESCRIBE HISTORY {T}").select("*"))

In [0]:
display(spark.sql(f"""
SELECT version,
       operationMetrics.numTargetRowsInserted  AS inserted,
       operationMetrics.numTargetRowsUpdated    AS updated,
       operationMetrics.numTargetFilesRemoved   AS files_removed,
       operationMetrics.numTargetFilesAdded     AS files_added
FROM (DESCRIBE HISTORY {T}) WHERE operation='MERGE'
"""))
# expect inserted=5, updated=5

In [0]:
try:
    df = (spark.read.format("delta")
          .option("startingVersion",2).option("endingVersion",2).table(T))
    df.display()
    print("Q1: no error (unexpected)")
except Exception as e:
    print("Q1 raised:", type(e).__name__, "|", str(e)[:200])

### CDF ON

In [0]:
T = "kafka_poc.engagement.cdf_on_poc"
base = "s3://retail-data-lake-dev/delta_poc/labs/cdf_on_poc"
spark.sql(f"DROP TABLE IF EXISTS {T}")
dbutils.fs.rm(base, recurse=True)

spark.sql(f"""
CREATE TABLE {T} (id INT, val STRING) USING DELTA LOCATION '{base}'
TBLPROPERTIES ('delta.enableChangeDataFeed'='True','delta.enableDeletionVectors'='True')
""")

# v1: 10 records
v1 = spark.createDataFrame(
    [(1,"A"),(2,"B"),(3,"C"),(4,"D"),(5,"E"),
     (6,"F"),(7,"G"),(8,"H"),(9,"I"),(10,"J")],
    "id INT, val STRING").repartition(1)
v1.writeTo(T).append()   # v1

# v2: 5 updates (id 1-5) + 5 inserts (id 11-15) via MERGE on id
src = spark.createDataFrame(
    [(1,"A2"),(2,"B2"),(3,"C2"),(4,"D2"),(5,"E2"),          # match -> update
     (11,"K"),(12,"L"),(13,"M"),(14,"N"),(15,"O")],         # no match -> insert
    "id INT, val STRING").createOrReplaceTempView("src")

spark.sql(f"""
MERGE INTO {T} AS t
USING src AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")   # v2



In [0]:
display(spark.sql(f"DESCRIBE HISTORY {T}").select("*"))

In [0]:
display(spark.sql(f"""
SELECT version,
       operationMetrics.numTargetRowsInserted  AS inserted,
       operationMetrics.numTargetRowsUpdated    AS updated,
       operationMetrics.numTargetFilesRemoved   AS files_removed,
       operationMetrics.numTargetFilesAdded     AS files_added
FROM (DESCRIBE HISTORY {T}) WHERE operation='MERGE'
"""))
# expect inserted=5, updated=5

In [0]:
from pyspark.sql.functions import col

In [0]:
try:
    df = (spark.read.format("delta").option("readChangeFeed","true")
          .option("startingVersion",2).option("endingVersion",2).table(T))
    df.filter(col('_change_type')!='update_preimage').display()
    print("Q1: no error (unexpected)")
except Exception as e:
    print("Q1 raised:", type(e).__name__, "|", str(e)[:200])

In [0]:
%sql
select * from kafka_poc.engagement.cdf_on_poc